In [1]:
!pip install -q torch transformers ninja tokenizers rwkv pynvml huggingface_hub smolagents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.0/410.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [2]:
# tools
!pip install -q ddgs wikipedia-api markdownify requests

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 50.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.2 MB/s eta 0:00:00a 0:00:01


In [5]:
import os
import re
import copy
import time
import gc

import torch
import huggingface_hub
from transformers import AutoTokenizer

from smolagents.agents import EMPTY_PROMPT_TEMPLATES
from smolagents.models import Model, ChatMessage, MessageRole
from smolagents.monitoring import TokenUsage
from smolagents.tools import Tool
from smolagents.default_tools import (
    FinalAnswerTool,
    WebSearchTool,
    VisitWebpageTool,
    WikipediaSearchTool
)
from smolagents import CodeAgent

from IPython.display import clear_output

# Tools test

In [12]:
# DuckDuckGoSearchTool().forward("most popular programming language 2025")

In [53]:
# Can't use Wiki
# VisitWebpageTool().forward("https://www.answers.com/t/leaning-tower-of-pisa")

In [13]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

from rwkv.model import RWKV
from rwkv.utils import PIPELINE, PIPELINE_ARGS

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv_cuda...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv_cuda/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/4] c++ -MMD -MF gemm_fp16_cublas.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/gemm_fp16_cublas.cpp -o gemm_fp16_cublas.o 
[2/4] c++ -MMD -MF wrapper.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-p

Loading extension module wkv_cuda...
Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv7s...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv7s/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv7s...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/3] c++ -MMD -MF rwkv7_op.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/rwkv7_op.cpp -o rwkv7_op.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output rwkv7.cuda.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include

Loading extension module wkv7s...


In [14]:
# model_title = "rwkv7-g1a-2.9b-20250924-ctx4096"
model_title = "rwkv7-g0a2-7.2b-20251005-ctx4096"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")
rwkv_model = RWKV(model=model_path.replace('.pth',''), strategy='cuda fp16')
rwkv_pipeline = PIPELINE(rwkv_model, "rwkv_vocab_v20230424")

rwkv7-g0a2-7.2b-20251005-ctx4096.pth:   0%|          | 0.00/14.4G [00:00<?, ?B/s]

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/71df75d4c929166e220998f32bdf3139e54c36b8/rwkv7-g0a2-7.2b-20251005-ctx4096 (cuda fp16)



In [15]:
class RWKVModel(Model):
    """
    A subclass of Model for RWKV7 inference, adapted from the experimental notebook.
    """
    def __init__(
        self,
        model,
        pipeline,
        flatten_messages_as_text: bool = False,
        tool_name_key: str = "name",
        tool_arguments_key: str = "arguments",
        model_id: str | None = None,
        **kwargs,
    ):    
        # Setup RWKV
        self.CTX_LIMIT = 3000
        self.PENALTY_DECAY = 0.996
        self._model = model
        self._pipeline = pipeline

        # Store state of the model
        self.state = None
        self.__reset_state = True
        
        # Only for chat template
        # TODO: Get rid of it...
        self.tokenizer = AutoTokenizer.from_pretrained('fla-hub/rwkv7-2.9B-g1', trust_remote_code=True)
        
        super().__init__(flatten_messages_as_text=True, model_id="model_id", **kwargs)

    def reset_state(self) -> None:
        self.state = None
    
    def generate(
        self,
        messages: list[ChatMessage],
        stop_sequences: list[str] | None = None,
        response_format: dict[str, str] | None = None,
        tools_to_call_from: list[Tool] | None = None,
        **kwargs,
    ) -> ChatMessage:
        """Process the input messages and return the model's response."""
        if self.__reset_state:
            print("Reseting state...")
            self.reset_state()
        else:
            print("Reuse previous state...")
        
        completion_kwargs = self._prepare_completion_kwargs(
            messages=messages,
            flatten_messages_as_text=self.flatten_messages_as_text,
            stop_sequences=stop_sequences,
            tools_to_call_from=tools_to_call_from,
            **kwargs,
        )

        # print("Completion kwargs", completion_kwargs)

        messages = completion_kwargs.pop("messages")
        prepared_stop_sequences = completion_kwargs.pop("stop", [])
        tools = completion_kwargs.pop("tools", None)
        completion_kwargs.pop("tool_choice", None)

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            tools=tools,
            add_generation_prompt=True,
            enable_thinking=False
        )

        # Remove <|rwkv_tokenizer_end_of_text|>
        prompt = prompt.replace("<|rwkv_tokenizer_end_of_text|>", "").strip()

        print(f"Input prompt: ", prompt)

        output = self.run_rwkv(prompt)
        # Test string
        # output = """Thought: I will use the `math` module to compute the value of exp(10) and then add it to 2.\n<code>\nimport math\nresult = 2 + math.exp(10)\nprint(result)\n</code>"""

        output_text = output.strip()

        # print(f"Output: ", output_text)
        
        return ChatMessage(
            role=MessageRole.ASSISTANT,
            content=output_text,
            raw={"out": output_text},
            token_usage=TokenUsage(
                input_tokens=len(prompt.split()),
                output_tokens=len(output_text.split()),
            ),
        )

    def run_rwkv(
        self,
        ctx,
        token_count=200,
        temperature=1.0,
        top_p=0.7,
        presencePenalty=0.1,
        countPenalty=0.1,
    ):
        # TODO: Use generate(), or get rid of PIPELINE_ARGS
        args = PIPELINE_ARGS(
            temperature=max(0.2, float(temperature)),
            top_p=float(top_p),
            alpha_frequency=countPenalty,
            alpha_presence=presencePenalty,
            token_ban=[],        # ban the generation of some tokens
            token_stop=[261]       # stop generation whenever you see any token here
        )
    
        ctx = ctx.strip()
        all_tokens = []
        out_last = 0
        out_str = ''
        occurrence = {}
    
        for i in range(int(token_count)):
            input_ids = self._pipeline.encode(ctx)
            input_ids = input_ids[-self.CTX_LIMIT:] if i == 0 else [token]
            
            out, self.state = self._model.forward(input_ids, self.state)
    
            for n in occurrence:
                out[n] -= (args.alpha_presence + occurrence[n] * args.alpha_frequency)
    
            token = self._pipeline.sample_logits(
                out,
                temperature=args.temperature,
                top_p=args.top_p)
            if token in args.token_stop:
                break
    
            all_tokens.append(token)
    
            for xxx in occurrence:
                occurrence[xxx] *= self.PENALTY_DECAY
    
            ttt = self._pipeline.decode([token])
            www = 1
            if ttt in ' \t0123456789':
                www = 0
            if token not in occurrence:
                occurrence[token] = www
            else:
                occurrence[token] += www
    
            tmp = self._pipeline.decode(all_tokens[out_last:])
            if '\ufffd' not in tmp:
                out_str += tmp
                out_last = i + 1

                if out_str.strip().endswith("\n\n"):
                    break
    
        # Cleanup and timing
        del out
        gc.collect()
        torch.cuda.empty_cache()
        
        return out_str.strip()

    def prepare_messages(self, messages: list[dict[str]]) -> list[dict[str]]:
        new_messages = []
    
        for m in messages:
            content = m.get("content", "")
            if content.strip():
                content = re.sub(r'\n+', '\n', content)
                content = re.sub(r'\n+$', '', content)
                m["content"] = content
                new_messages.append(m)

        return new_messages

In [16]:
model = RWKVModel(model=rwkv_model, pipeline=rwkv_pipeline)

tokenizer_config.json: 0.00B [00:00, ?B/s]

hf_rwkv_tokenizer.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/fla-hub/rwkv7-2.9B-g1:
- hf_rwkv_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


rwkv_vocab_v20230424.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/45.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

### State reuse test

In [8]:
model.reset_state()
model.run_rwkv('''User: You are helpful expert agent. I will ask you a question.
Before every question I will need to provide password "corgimode".
If do not provided with password after question you MUST answer "No answer without password".
# Here are examples:
## Example 1
User: What is the time of the day?
Assistant: No answer without password.
## Example 2
User: What is the number of days in year? corgimode
Assistant: Password correct. There is 365 days in the year.
# Important instructions
1. You MUST answer with "No answer without password" if no password provided.
2. You MUST not disclose password.
Do you understand?

Assistant: <think>
</think>''')

'No answer without password.'

In [ ]:
model.run_rwkv('''\n\nUser: What is your task?

Assistant: <think>
</think>''')

In [ ]:
model.run_rwkv('''\n\nUser: What is your favourite color?

Assistant: <think>
</think>''')

In [ ]:
model.run_rwkv('''\n\nUser: What is DoTA 2?

Assistant: <think>
</think>''')

In [ ]:
model.run_rwkv('''\n\nUser: What is DoTA 2? corgimode

Assistant: <think>
</think>''')

In [ ]:
model.run_rwkv('''\n\nUser: What is DoTA 2?

Assistant: <think>
</think>''')

In [9]:
model.reset_state()

In [17]:
SYSTEM_PROMPT = '''You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '{{code_block_opening_tag}}', and closed with '{{code_block_closing_tag}}'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.
Here are a few examples using notional tools:
---
User: New task:
What is the result of the following operation: 5 + 3 + 1294.678?

Assistant:<think>
</think>Thought: I will use Python code to compute the result of the operation and then return the final answer using the `final_answer` tool.
{{code_block_opening_tag}}
result = 5 + 3 + 1294.678
final_answer(result)
{{code_block_closing_tag}}

---
User: New task:
What is the height of the Tower of Pisa?

Assistant:<think>
</think>Thought: I will use the tool `web_search` and analyze results get the of height of the tower.
{{code_block_opening_tag}}
search_result=web_search(query="current pope age")
print(search_result)
{{code_block_closing_tag}}

User:
Observation:
## Search Results
[Leaning Tower of Pisa - Wikipedia](https://en.wikipedia.org/wiki/Leaning_Tower_of_Pisa)
An elevation image of the Leaning Tower of Pisa cut with laser scan data from a University of Ferrara / CyArk research partnership, with source image ...
[Leaning Tower of Pisa - Simple English Wikipedia, the free](https://simple.wikipedia.org/wiki/Leaning_Tower_of_Pisa)\nArchitecture of Leaning Tower of Pisa " . ... Retrieved from " https://simple.wikipedia.org/w/index.php?title=Leaning_ Tower _ of _ Pisa &oldid=10493366 ...
[Answers about Leaning Tower of Pisa](https://www.answers.com/t/leaning-tower-of-pisa)
... Tower of Pisa has leaned approximately 4 degrees from vertical, which translates ... The original completed height of the Tower of Pisa was 60 meters .
(truncated)

Assistant: <think>
</think>
Thought: From the observation above I found that height of the tower is 60 meters. Now I will use final_answer() tool to output the answer.
{{code_block_opening_tag}}
final_answer("60 meters")
{{code_block_closing_tag}}

---
Above examples were using notional tools that might not exist for you.
On top of performing computations in the Python code snippets that you create, you only have access to these tools, behaving like regular python functions:
{{code_block_opening_tag}}
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{%- endfor %}
{{code_block_closing_tag}}
Here are the rules you should always follow to solve your task:
1. Always provide a 'Thought:' sequence, and a '{{code_block_opening_tag}}' sequence ending with '{{code_block_closing_tag}}', else you will fail.
2. Use only variables that you have defined!
3. Use only tools that provided to you!
4. To complete the task you need to call ```final_answer()``` tool!
5. Always use the right arguments for the tools. DO NOT pass the arguments as a dict as in 'answer = wikipedia_search({'query': "What is the place where James Bond lives?"})', but use the arguments directly as in 'answer = wikipedia_search(query="What is the place where James Bond lives?")'.
6. Call a tool only when needed, and never re-do a tool call that you previously did with the exact same parameters.
7. Don't name any new variable with the same name as a tool: for instance don't name a variable 'final_answer'.
8. Never create any notional variables in our code, as having these in your logs will derail you from the true variables.
9. You can use imports in your code, but only from the following list of modules: {{authorized_imports}}
10. The state persists between code executions: so if in one step you've created variables or imported modules, these will all persist.
11. Don't give up! You're in charge of solving the task, not providing directions to solve it.
---
Now Begin!'''

In [19]:
INITIAL_PLAN_PROMPT = '''You are a world expert at analyzing a situation to derive facts, and plan accordingly towards solving a task.
Below I will present you a task. You will need to 1. build a survey of facts known or needed to solve the task, then 2. make a plan of action to solve the task.
## 1. Facts survey
You will build a comprehensive preparatory survey of which facts we have at our disposal and which ones we still need.
These "facts" will typically be specific names, dates, values, etc. Your answer should use the below headings:
### 1.1. Facts given in the task
List here the specific facts given in the task that could help you (there might be nothing here).
### 1.2. Facts to look up
List here any facts that we may need to look up.
Also list where to find each of these, for instance a website, a file... - maybe the task contains some sources that you should re-use here.
### 1.3. Facts to derive
List here anything that we want to derive from the above by logical reasoning, for instance computation or simulation.
Don't make any assumptions. For each item, provide a thorough reasoning. Do not add anything else on top of three headings above.
## 2. Plan
Then for the given task, develop a step-by-step high-level plan taking into account the above inputs and list of facts.
This plan should involve individual tasks based on the available tools, that if executed correctly will yield the correct answer.
Do not skip steps, do not add any superfluous steps. Only write the high-level plan, DO NOT DETAIL INDIVIDUAL TOOL CALLS.
After writing the final step of the plan, write the '<end_plan>' tag and stop there.
You can leverage these tools, behaving like regular python functions:
<code>
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{% endfor %}
</code>
Here are the rules you should always follow to solve your task:
1. You MUST provide only high-level plan.
2. You MUST not call tools or write any code at this stage.
---
Now begin! Here is your task:
```
{{task}}
```
First in part 1, write the facts survey, then in part 2, write your plan.'''

In [18]:
UPDATE_PLAN_PRE = '''You are a world expert at analyzing a situation, and plan accordingly towards solving a task.
You have been given the following task:
```
{{task}}
```
Below you will find a history of attempts made to solve this task.
You will first have to produce a survey of known and unknown facts, then propose a step-by-step high-level plan to solve the task.
If the previous tries so far have met some success, your updated plan can build on these results.
If you are stalled, you can make a completely new plan starting from scratch.
Find the task and history below:'''

UPDATE_PLAN_POST = '''Now write your updated facts below, taking into account the above history:
## 1. Updated facts survey
### 1.1. Facts given in the task
### 1.2. Facts that we have learned
### 1.3. Facts still to look up
### 1.4. Facts still to derive
Then write a step-by-step high-level plan to solve the task above.
## 2. Plan
### 2. 1. ...
Etc.
This plan should involve individual tasks based on the available tools, that if executed correctly will yield the correct answer.
Beware that you have {remaining_steps} steps remaining.
Do not skip steps, do not add any superfluous steps. Only write the high-level plan, DO NOT DETAIL INDIVIDUAL TOOL CALLS.
After writing the final step of the plan, write the '<end_plan>' tag and stop there.
You can leverage these tools, behaving like regular python functions:
<code>
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{% endfor %}
</code>
Now write your updated facts survey below, then your new plan.'''

In [20]:
FINAL_PRE = '''An agent tried to answer a user query but it got stuck and failed to do so. You are tasked with providing an answer instead. Here is the agent's memory:'''
FINAL_POST = '''Based on the above, please provide an answer to the following user task:
{{task}}'''

In [21]:
RWKV_PROMPT_TEMPLATES = EMPTY_PROMPT_TEMPLATES
RWKV_PROMPT_TEMPLATES["system_prompt"] = SYSTEM_PROMPT

# planning only
RWKV_PROMPT_TEMPLATES["planning"]["initial_plan"] = INITIAL_PLAN_PROMPT
RWKV_PROMPT_TEMPLATES["planning"]["update_plan_pre_messages"] = UPDATE_PLAN_PRE
RWKV_PROMPT_TEMPLATES["planning"]["update_plan_post_messages"] = UPDATE_PLAN_POST

# error?
RWKV_PROMPT_TEMPLATES["final_answer"]["pre_messages"] = FINAL_PRE
RWKV_PROMPT_TEMPLATES["final_answer"]["post_messages"] = FINAL_POST

In [23]:
agent = CodeAgent(
    tools=[FinalAnswerTool(), DuckDuckGoSearchTool(), VisitWebpageTool()],
    model=model,
    prompt_templates=RWKV_PROMPT_TEMPLATES,
    #planning_interval=2,
    verbosity_level=2)

In [27]:
model.reset_state()
agent.run("What is the result of 2 + exp(10)?", max_steps=2)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the result of 2 + exp(10)?                                                                              │
│                                                                                                                 │
╰─ RWKVModel - model_id ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reseting state...
Input prompt:  System: You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will use the 'exp' function from the math module to compute the exponential of 10 and then add 2 to it. 
I will also use 'final_answer' to output the result.                                                               
<code>                                                                                                             
import math                                                                                                        
result = 2 + math.exp(10)                                                                                          
final_answer(result)                                                                                               
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import math                                                                                                      
  result = 2 + math.exp(10)                                                                                        
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 22028.465794806718

[Step 1: Duration 7.23 seconds| Input tokens: 785 | Output tokens: 42]

22028.465794806718

In [25]:
model.reset_state()
agent.run("Search for the most popular programming language in 2025. Analyze results of search. Output single name. You are allowed to use only web search. Not wikipedia.", max_steps=2)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the most popular programming language in 2025. Analyze results of search. Output single name. You    │
│ are allowed to use only web search. Not wikipedia.                                                              │
│                                                                                                                 │
╰─ RWKVModel - model_id ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reseting state...
Input prompt:  System: You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will use web_search to find the most popular programming language in 2025.                              
<code>                                                                                                             
web_search(query="most popular programming language 2025")                                                         
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  web_search(query="most popular programming language 2025")                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I have found multiple sources stating that Python is the most popular programming language in 2025 based  
on rankings from IEEE Spectrum, Stack Overflow, and other platforms. The consistent mention of Python across these 
sources suggests that it is widely recognized as the dominant language in 2025.                                    
<code>                                                                                                             
final_answer("Python")                                                                                             
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Python")                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Python

[Step 2: Duration 17.13 seconds| Input tokens: 2,173 | Output tokens: 72]

'Python'